# Advanced Retrieval Techniques

This notebook demonstrates advanced retrieval techniques for RAG: Query Rewriting, HyDE, and Cross-Encoder Reranking.

## What You'll Learn

1. **Query Rewriting** - Transform queries to improve retrieval
2. **HyDE** - Hypothetical Document Embeddings
3. **Cross-Encoder Reranking** - Improve result quality with second-stage reranking
4. **Combining Techniques** - Building a production-ready pipeline

## Why Advanced Techniques?

Basic retrieval has limitations:

| Issue | Description | Solution |
|-------|-------------|----------|
| Vocabulary Mismatch | User says car but docs say automobile | Query rewriting |
| Semantic Gap | Query and docs use different phrasing | HyDE |
| Recall vs Precision | Too many or too few results | Reranking

## Architecture Overview

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  Query   │───▶│  Rewrite │───▶│ Retrieve │───▶│ Rerank   │
└──────────┘    └──────────┘    └──────────┘    └────┬─────┘
                                                     │
                                                     ▼
                                              ┌──────────┐
                                              │   Top-k  │
                                              │ Results  │
                                              └──────────┘
```

## Techniques Covered

1. **Query Rewriting** - Transform queries to improve retrieval
2. **HyDE** - Hypothetical Document Embeddings
3. **Cross-Encoder Reranking** - Improve result quality with second-stage reranking

## Prerequisites

Install Ollama and pull the required models:
```bash
ollama pull llama3.2
ollama pull nomic-embed-text
```

In [ ]:
# Install dependencies
!pip install langchain langchain-community langchain-ollama chromadb

In [2]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

## 1. Setup Documents

In [3]:
# Sample documents about RAG
documents = [
    Document(
        page_content="""Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowledge bases. This helps the model generate more accurate and up-to-date responses.""",
        metadata={"source": "doc1", "topic": "intro"}
    ),
    Document(
        page_content="""The basic RAG pipeline consists of three components: 1) Retriever that finds relevant documents using embeddings and vector similarity search, 2) Database storing document embeddings, and 3) Generator that produces final output using retrieved context.""",
        metadata={"source": "doc2", "topic": "architecture"}
    ),
    Document(
        page_content="""Embedding models convert text into numerical vectors (embeddings) that capture semantic meaning. Popular embedding models include OpenAI's text-embedding-3-small, Cohere, and open-source options like nomic-embed-text.""",
        metadata={"source": "doc3", "topic": "embeddings"}
    ),
    Document(
        page_content="""Vector databases like Chroma, Pinecone, Weaviate, and Milvus store embeddings and enable fast similarity search. They typically use algorithms like HNSW or IVF for efficient nearest neighbor search.""",
        metadata={"source": "doc4", "topic": "vector-db"}
    ),
    Document(
        page_content="""HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge the gap between query language and document language.""",
        metadata={"source": "doc5", "topic": "hyde"}
    ),
    Document(
        page_content="""Query rewriting improves retrieval by transforming user queries before searching. This helps when users ask questions in ways that don't directly match how information is stored in the knowledge base.""",
        metadata={"source": "doc6", "topic": "query-rewrite"}
    ),
    Document(
        page_content="""Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-orders the results for better relevance.""",
        metadata={"source": "doc7", "topic": "reranking"}
    ),
    Document(
        page_content="""Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy.""",
        metadata={"source": "doc8", "topic": "self-rag"}
    )
]

print(f"Loaded {len(documents)} documents")

Loaded 8 documents


In [4]:
# Create vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="advanced_rag_demo"
)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store created!")

Vector store created!


## 2. Baseline Retrieval (No Advanced Techniques)

In [5]:
def display_results(query, results, title="Results"):
    """Display retrieval results."""
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"\n{title}:")
    print("-" * 60)
    for i, doc in enumerate(results, 1):
        print(f"\n{i}. [{doc.metadata.get('source', 'unknown')}] {doc.page_content[:150]}...")

# Test baseline
query = "How do I improve my RAG system?"
baseline_results = base_retriever.invoke(query)
display_results(query, baseline_results, "Baseline Retrieval")


Query: How do I improve my RAG system?

Baseline Retrieval:
------------------------------------------------------------

1. [doc1] Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowle...

2. [doc7] Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-ord...

3. [doc8] Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy....

4. [doc5] HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge t...


## 3. Query Rewriting

**What:** Transforming user queries to better match document language.

**Why:** Users often ask questions differently than how information is written in documents.

## The Vocabulary Mismatch Problem

```
User Query: "How do I fix my car?"
                 │
                 ▼
        ┌────────────────┐
        │ Vector Search  │
        └────────────────┘
                 │
                 ▼
Document: "Vehicle maintenance procedures..."

Problem: "car" ≠ "vehicle" in vector space!
```

## Query Rewriting Approaches

| Approach | Description | Pros | Cons |
|----------|-------------|------|------|
| LLM Rewrite | Use LLM to rewrite query | High quality | Slower, costs more |
| Multi-Query | Generate multiple versions | Better coverage | More queries |
| HyDE | Generate hypothetical answer | Semantic match | Can hallucinate

In [6]:
# Initialize LLM
llm = ChatOllama(model="llama3.2")

In [7]:
class QueryRewriter:
    """Rewrite queries for better retrieval."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def rewrite(self, query: str) -> str:
        """Rewrite a single query."""
        prompt = f"""Rewrite this query to improve document retrieval.
        Keep it as a question, but use different words and make it more explicit.

Original: {query}

Rewritten:"""
        response = self.llm.invoke(prompt)
        return response.content.strip()
    
    def multi_query(self, query: str) -> list:
        """Generate multiple query variations."""
        prompt = f"""Generate 3 different versions of this query that might
        retrieve relevant documents. Consider different phrasings and synonyms.

Original: {query}

Queries:"""
        response = self.llm.invoke(prompt)
        
        # Parse response
        lines = response.content.strip().split('\n')
        queries = [query]  # Include original
        for line in lines:
            line = line.strip()
            if line and (line[0].isdigit() or line.startswith('-')):
                q = line.lstrip('0123456789.-').strip()
                if q:
                    queries.append(q)
        return queries[:4]

In [8]:
# Test query rewriting
rewriter = QueryRewriter(llm)

query = "How do I improve my RAG system?"
rewritten = rewriter.rewrite(query)
print(f"Original: {query}")
print(f"Rewritten: {rewritten}")

Original: How do I improve my RAG system?
Rewritten: How can I optimize the performance and efficiency of my Relational Answering Generator (RAG) system?


In [9]:
# Test multi-query
multi_queries = rewriter.multi_query(query)
print(f"\nMultiple queries generated:")
for i, q in enumerate(multi_queries, 1):
    print(f"  {i}. {q}")


Multiple queries generated:
  1. How do I improve my RAG system?
  2. **Phrased query with synonyms**:
  3. **Query with specific components**:
  4. **Query with a broader context**:


In [10]:
# Multi-query retrieval
class MultiQueryRetriever:
    """Retrieve using multiple query variations."""
    
    def __init__(self, base_retriever, llm):
        self.base_retriever = base_retriever
        self.rewriter = QueryRewriter(llm)
    
    def retrieve(self, query: str, k: int = 4) -> list:
        """Retrieve with multiple queries."""
        queries = self.rewriter.multi_query(query)
        
        all_docs = []
        for q in queries:
            docs = self.base_retriever.invoke(q)
            all_docs.extend(docs)
        
        # Deduplicate
        seen = set()
        unique = []
        for doc in all_docs:
            key = doc.page_content[:50]
            if key not in seen:
                seen.add(key)
                unique.append(doc)
        
        return unique[:k]

In [11]:
# Test with query rewriting
mq_retriever = MultiQueryRetriever(base_retriever, llm)
query_rewrite_results = mq_retriever.retrieve(query)
display_results(query, query_rewrite_results, "With Query Rewriting")


Query: How do I improve my RAG system?

With Query Rewriting:
------------------------------------------------------------

1. [doc1] Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowle...

2. [doc7] Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-ord...

3. [doc8] Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy....

4. [doc5] HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge t...


## 4. HyDE (Hypothetical Document Embeddings)

**What:** Generating a hypothetical ideal answer document and using it for retrieval.

**Why:** Instead of searching with the query, search with what the answer MIGHT look like.

## How HyDE Works

```
Query: "How do I improve my RAG system?"
            │
            ▼
┌─────────────────────────────────────┐
│  LLM generates hypothetical answer: │
│                                     │
│  "To improve your RAG system,       │
│   consider query rewriting,         │
│   better chunking strategies,       │
│   and reranking..."                 │
└─────────────────────────────────────┘
            │
            ▼
   Embed the hypothetical answer
            │
            ▼
   Find documents similar to the answer!
```

## Why It Works

- The hypothetical answer uses **document-style language**
- It bridges the gap between **query language** and **document language**
- Even if the hypothetical is wrong, similar documents are often retrieved

**Trade-off:** HyDE can sometimes retrieve irrelevant docs if the hypothesis is off. Use with caution for factual queries.

In [12]:
class HyDERetriever:
    """HyDE: Hypothetical Document Embeddings."""
    
    def __init__(self, vectorstore, llm, embeddings):
        self.embeddings = embeddings
        self.vectorstore = vectorstore
        self.llm = llm
    
    def generate_hypothetical(self, query: str) -> str:
        """Generate a hypothetical answer document."""
        prompt = f"""Write a detailed hypothetical document that could answer this question.
        Include relevant facts and explanations.

Question: {query}

Hypothetical Document:"""
        response = self.llm.invoke(prompt)
        return response.content
    
    def retrieve(self, query: str, k: int = 4) -> list:
        """Retrieve using HyDE."""
        # Generate hypothetical document
        hypothetical = self.generate_hypothetical(query)
        print(f"Hypothetical document: {hypothetical[:200]}...")
        
        query_emb = self.embeddings.embed_query(query)
        hypo_emb = self.embeddings.embed_query(hypothetical)
        
        # Combine embeddings (average)
        combined_emb = [(q + h) / 2 for q, h in zip(query_emb, hypo_emb)]
        
        # Search
        results = self.vectorstore.similarity_search_by_vector(
            combined_emb,
            k=k
        )
        
        return results

In [13]:
# Test HyDE
hyde_retriever = HyDERetriever(vectorstore, llm, embeddings)
hyde_results = hyde_retriever.retrieve(query)
display_results(query, hyde_results, "With HyDE")

Hypothetical document: **Improving Your RAG (Risk, Action, Goal) System: A Comprehensive Guide**

**Introduction**

The RAG system is a widely used framework for managing and achieving goals. It involves identifying and ass...

Query: How do I improve my RAG system?

With HyDE:
------------------------------------------------------------

1. [doc1] Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowle...

2. [doc8] Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy....

3. [doc7] Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-ord...

4. [doc5] HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge t...


## 5. LLM Reranking

**What:** Using an LLM (like llama3.2) to score and re-rank initial retrieval results.

**Why:** We can use a powerful local LLM to determine relevance instead of a specialized cross-encoder model.

| Aspect | Bi-Encoder | LLM Reranker |
|--------|------------|--------------|
| Speed | Fast (pre-computed) | Medium (model-dependent) |
| Quality | Good | Excellent (via LLM) |
| Use Case | Initial retrieval | Reranking

## The Reranking Pipeline

```
Step 1: Initial Retrieval (Bi-Encoder)
         Get top-20 results (fast)
                  │
                  ▼
Step 2: Prepare Query-Document Pairs
         [[query, doc1], [query, doc2], ...]
                  │
                  ▼
Step 3: Cross-Encoder Scoring
         Score each pair for relevance
                  │
                  ▼
Step 4: Re-rank by Score
         Return top-4 results
```

> **This notebook uses:** `ms-marco-MiniLM-L-6-v2` - a fast, effective cross-encoder trained on search data.

In [14]:
# Initialize Ollama reranker
class OllamaReranker:
    """Rerank using Ollama LLM to score query-document relevance."""
    
    def __init__(self, base_retriever, llm, top_k: int = 20):
        self.base_retriever = base_retriever
        self.llm = llm
        self.top_k = top_k
    
    def rerank(self, query: str, final_k: int = 4) -> list:
        """Rerank documents using LLM scoring."""
        # Initial retrieval
        initial_docs = self.base_retriever.invoke(query)
        initial_docs = initial_docs[:self.top_k]
        
        # Score each document using LLM
        doc_scores = []
        for doc in initial_docs:
            score = self._score_document(query, doc.page_content)
            doc_scores.append((doc, score))
        
        # Sort by score
        doc_scores.sort(key=lambda x: x[1], reverse=True)
        
        return doc_scores[:final_k]
    
    def _score_document(self, query: str, document: str) -> float:
        """Score a document's relevance to a query using LLM."""
        prompt = f"""Rate the relevance of the document to the query on a scale of 1-10.
Only respond with a single number.

Query: {query}

Document: {document}

Relevance score (1-10):"""
        response = self.llm.invoke(prompt)
        try:
            # Extract first number from response
            import re
            numbers = re.findall(r'-?\d+\.?\d*', response.content)
            if numbers:
                score = float(numbers[0])
                # Clamp to 1-10 range
                return max(1.0, min(10.0, score))
        except:
            pass
        return 5.0  # Default neutral score

reranker = OllamaReranker(base_retriever, llm)
print("Ollama Reranker initialized!")

Ollama Reranker initialized!


In [15]:
# Test reranking
reranked_results = reranker.rerank(query)

print("\nResults with Ollama Reranking:")
print("=" * 60)
for i, (doc, score) in enumerate(reranked_results, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   {doc.page_content[:150]}...")


Results with Ollama Reranking:

1. Score: 9.0000
   Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowle...

2. Score: 6.0000
   Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-ord...

3. Score: 6.0000
   Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy....

4. Score: 6.0000
   HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge t...


## 6. Complete Advanced RAG Pipeline

In [16]:
class AdvancedRAGPipeline:
    """Complete advanced RAG pipeline with all techniques."""
    
    def __init__(self, vectorstore, llm, reranker):
        self.vectorstore = vectorstore
        self.llm = llm
        self.reranker = reranker
        
        self.base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})
        self.query_rewriter = QueryRewriter(llm)
        self.hyde = HyDERetriever(vectorstore, llm, embeddings)
        self.ollama_reranker = OllamaReranker(self.base_retriever, llm)
    
    def query(self, question: str, use_all_techniques: bool = True) -> dict:
        """Query the RAG system."""
        
        if use_all_techniques:
            # Step 1: Query rewriting
            rewritten = self.query_rewriter.rewrite(question)
            
            # Step 2: Multi-query retrieval
            mq_retriever = MultiQueryRetriever(self.base_retriever, self.llm)
            initial_docs = mq_retriever.retrieve(question, k=10)
            
            # Step 3: Reranking using Ollama LLM
            doc_scores = []
            for doc in initial_docs:
                score = self.ollama_reranker._score_document(question, doc.page_content)
                doc_scores.append((doc, score))
            doc_scores.sort(key=lambda x: x[1], reverse=True)
            final_docs = [doc for doc, _ in doc_scores[:4]]
            
            return {
                "rewritten_query": rewritten,
                "documents": final_docs
            }
        else:
            # Baseline
            docs = self.base_retriever.invoke(question)
            return {
                "rewritten_query": question,
                "documents": docs[:4]
            }

In [17]:
# Test complete pipeline
advanced_rag = AdvancedRAGPipeline(vectorstore, llm, reranker)

query = "How do I improve my RAG system?"
result = advanced_rag.query(query)

print(f"Original query: {query}")
print(f"Rewritten query: {result['rewritten_query']}")
print(f"\nFinal retrieved documents: {len(result['documents'])}")
for i, doc in enumerate(result['documents'], 1):
    print(f"\n{i}. {doc.page_content[:150]}...")

Original query: How do I improve my RAG system?
Rewritten query: Here's a rewritten version of the query:

What strategies can I implement to enhance the effectiveness and efficiency of my Retrieval and Access Gateway (RAG) system?

This rewritten query is more explicit and asks for specific solutions, rather than just a general improvement.

Final retrieved documents: 4

1. Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowle...

2. Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-ord...

3. Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy....

4. HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge t.

## 7. Comparison Summary

In [18]:
# Compare all techniques
query = "How do I improve my RAG system?"

print("=" * 70)
print("COMPARISON: Advanced Retrieval Techniques")
print("=" * 70)

print("\n1. BASELINE RETRIEVAL")
baseline = base_retriever.invoke(query)
print(f"   Retrieved {len(baseline)} documents")
print(f"   First result: {baseline[0].metadata.get('source', 'unknown')}")

print("\n2. QUERY REWRITING")
mq_result = mq_retriever.retrieve(query)
print(f"   Retrieved {len(mq_result)} unique documents")
print(f"   First result: {mq_result[0].metadata.get('source', 'unknown')}")

print("\n3. HYDE")
hyde_result = hyde_retriever.retrieve(query)
print(f"   Retrieved {len(hyde_result)} documents")
print(f"   First result: {hyde_result[0].metadata.get('source', 'unknown')}")

print("\n4. OLLAMA RERANKING (llama3.2)")
reranked = reranker.rerank(query)
print(f"   Retrieved {len(reranked)} documents (from top 20)")
print(f"   First result: {reranked[0][0].metadata.get('source', 'unknown')}")

print("\n" + "=" * 70)

COMPARISON: Advanced Retrieval Techniques

1. BASELINE RETRIEVAL
   Retrieved 4 documents
   First result: doc1

2. QUERY REWRITING
   Retrieved 4 unique documents
   First result: doc1

3. HYDE
Hypothetical document: **Improving Your RAG (Read, Act, Grow) System: A Comprehensive Guide**

**Introduction**

The RAG system is a popular framework for setting and achieving goals, particularly in personal and profession...
   Retrieved 4 documents
   First result: doc1

4. OLLAMA RERANKING (llama3.2)
   Retrieved 4 documents (from top 20)
   First result: doc1



## Summary

This notebook covered three key advanced retrieval techniques:

1. **Query Rewriting**: Transform queries to better match document language

2. **HyDE**: Use hypothetical documents to improve semantic matching

3. **Cross-Encoder Reranking**: Second-stage re-scoring for better precision

## When to Use What

| Technique | Best For | Complexity | Latency |
|-----------|----------|------------|---------|
| Query Rewriting | Varied phrasing | Medium | +100-300ms |
| HyDE | Conceptual queries | Medium | +200-500ms |
| Reranking | Precision critical | Low | +50-100ms |
| Combined | Production | Higher | +300-900ms |

## Decision Guide

Start with baseline. If quality is good, done. Otherwise:
- Precision matters more than speed -> add reranking
- Users phrase differently than docs -> add query rewriting
- Semantic matching is poor -> add HyDE

## Trade-off Summary

| Technique | Gains | Costs |
|-----------|-------|-------|
| Query Rewriting | Better recall | Extra LLM call |
| HyDE | Better semantic match | LLM generation per query |
| Reranking | Better precision | Cross-encoder call |
| Multi-Query | Broader coverage | Multiple vector searches |

## Additional Resources

- [HyDE Paper](https://arxiv.org/abs/2212.10496)
- [Cross-Encoder Docs](https://sbert.net/examples/cross_encoder/)